In [ ]:
import os
import pickle
import jsonlines
import pandas as pd
import numpy as np
import json
import copy
from tqdm import tqdm
from collections import defaultdict
import requests

In [ ]:
id_map = json.load(open("./handled/id_map.json"))
item_dict = json.load(open("./handled/item2attributes.json", "r"))

In [ ]:
def load_dataset():
        '''Load train, validation, test dataset'''

        usernum = 0
        itemnum = 0
        User = defaultdict(list)    # default value is a blank list
        user_train = {}
        user_valid = {}
        user_test = {}
        # assume user/item index starting from 1
        f = open('./handled/inter.txt', 'r')
        for line in f:  # use a dict to save all seqeuces of each user
            u, i = line.rstrip().split(' ')
            u = int(u)
            i = int(i)
            usernum = max(u, usernum)
            itemnum = max(i, itemnum)
            User[u].append(i)

        for user in tqdm(User):
            nfeedback = len(User[user])
            #nfeedback = len(User[user])
            if nfeedback < 3:
                user_train[user] = User[user]
                user_valid[user] = []
                user_test[user] = []
            else:
                user_train[user] = User[user][:-2]
                user_valid[user] = []
                user_valid[user].append(User[user][-2])
                user_test[user] = []
                user_test[user].append(User[user][-1])
        
        return user_train
        

In [ ]:
inter = load_dataset()

In [ ]:
prompt_template = "The user has visited following fashions: \n<HISTORY> \nplease conclude the user's perference."

In [ ]:
def get_user_prompt(history):

    user_str = copy.deepcopy(prompt_template)
    hist_str = ""
    for item in history:
        try:    # some item does not have title
            item_str = item_dict[id_map["id2item"][str(item)]]["title"]
            hist_str = hist_str + item_str + ", "
        except:
            continue

    # limit the prompt length
    if len(hist_str) > 8000:
        hist_str = hist_str[-8000:]

    user_str = user_str.replace("<HISTORY>", hist_str)

    return user_str

In [ ]:
user_data = {}

for user, history in tqdm(inter.items()):
    user_data[user] = get_user_prompt(history)

In [ ]:
# json.dump(user_data, open("./handled/user_str.json", "w"))

In [ ]:
user_data = json.load(open("./handled/user_str.json", "r"))

In [ ]:
# Configure the embedding endpoint without storing credentials in this notebook.
# Set these environment variables before starting Jupyter.
import os
import requests
import time

EMBEDDING_URL = os.environ.get("CSBREC_EMBEDDING_URL", "")
EMBEDDING_API_KEY = os.environ.get("CSBREC_EMBEDDING_API_KEY", "")
if not EMBEDDING_URL or not EMBEDDING_API_KEY:
    raise RuntimeError(
        "Set CSBREC_EMBEDDING_URL and CSBREC_EMBEDDING_API_KEY before running this notebook."
    )


In [ ]:
def get_response(prompt):
    # Preserve the embedding model used by the upstream preprocessing pipeline.
    payload = {"model": "text-embedding-ada-002", "input": prompt}
    headers = {"Authorization": f"Bearer {EMBEDDING_API_KEY}"}
    for attempt in range(3):
        try:
            response = requests.post(
                EMBEDDING_URL, headers=headers, json=payload, timeout=60
            )
            response.raise_for_status()
            return response.json()["data"][0]["embedding"]
        except requests.RequestException as exc:
            status = exc.response.status_code if exc.response is not None else None
            # Retry transient failures only, and let the caller save its cache.
            if attempt == 2 or (status is not None and status != 429 and status < 500):
                raise
            time.sleep(2 ** attempt)


In [ ]:
cache_path = "./handled/user_emb.pkl"
if os.path.exists(cache_path):
    with open(cache_path, "rb") as cache_file:
        user_emb = pickle.load(cache_file)
else:
    user_emb = {}


In [ ]:
len(user_emb)

In [ ]:
try:
    for key, value in tqdm(user_data.items()):
        if key not in user_emb:
            if len(value) > 4096:
                value = value[:4095]
            user_emb[key] = get_response(value)
finally:
    # Persist completed requests on success, failure or keyboard interruption.
    with open(cache_path, "wb") as cache_file:
        pickle.dump(user_emb, cache_file)


In [ ]:
len(user_emb)

In [ ]:
emb_list = []
for key, value in tqdm(user_emb.items()):
    emb_list.append(value)

emb_list = np.array(emb_list)
pickle.dump(emb_list, open("./handled/usr_emb_np.pkl", "wb"))